# Lesson 3.3 - Model Evaluation & Selection

## Objectives
- Compare models with stratified cross-validation.
- Run hyperparameter tuning with leakage-safe setup.
- Select final model using test-set evaluation and business constraints.
## How to run (recommended)
- Restart kernel -> Run all.
- Do not touch the test set until the end (use it once).

## Verify
- Add one explicit leakage check to your workflow.
- Write down the metric + threshold you would use as a release gate.


In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    XGBClassifier = None

print("xgboost_available:", HAS_XGB)

xgboost_available: True


## Section A - Split and Leakage-Safe Candidate Pipelines

In [2]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    "log_reg": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000)),
        ]
    ),
    "random_forest": RandomForestClassifier(n_estimators=250, random_state=42),
    "grad_boost": GradientBoostingClassifier(random_state=42),
}
if HAS_XGB:
    models["xgboost"] = XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
    )

## Section B - Cross-Validation Comparison

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"auc": "roc_auc", "f1": "f1", "acc": "accuracy"}

rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=None)
    rows.append(
        {
            "model": name,
            "cv_auc_mean": scores["test_auc"].mean(),
            "cv_auc_std": scores["test_auc"].std(),
            "cv_f1_mean": scores["test_f1"].mean(),
            "cv_acc_mean": scores["test_acc"].mean(),
        }
    )

cv_df = pd.DataFrame(rows).sort_values("cv_auc_mean", ascending=False)
display(cv_df)

,model,cv_auc_mean,cv_auc_std,cv_f1_mean,cv_acc_mean
0,log_reg,0.996151,0.004044,0.983314,0.978878
3,xgboost,0.992038,0.005362,0.973897,0.967196
1,random_forest,0.988422,0.010684,0.970127,0.962462
2,grad_boost,0.987314,0.009289,0.968182,0.960137


## Section C - Hyperparameter Tuning

We tune Random Forest using both grid and random search.

In [4]:
rf_grid = {
    "n_estimators": [150, 250, 400],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
}

rf_grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
)
rf_grid_search.fit(X_train, y_train)

rf_rand = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_grid,
    n_iter=8,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
rf_rand.fit(X_train, y_train)

print("grid best:", rf_grid_search.best_params_, "score:", round(rf_grid_search.best_score_, 4))
print("random best:", rf_rand.best_params_, "score:", round(rf_rand.best_score_, 4))

grid best: {'max_depth': 5, 'min_samples_split': 10, 'n_estimators': 150} score: 0.9892
random best: {'n_estimators': 250, 'min_samples_split': 5, 'max_depth': 5} score: 0.9891


## Section D - Final Test Evaluation

In [5]:
final_models = {
    "log_reg": models["log_reg"],
    "rf_tuned": rf_grid_search.best_estimator_,
    "grad_boost": models["grad_boost"],
}
if HAS_XGB:
    final_models["xgboost"] = models["xgboost"]

rows = []
for name, model in final_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append(
        {
            "model": name,
            "test_auc": roc_auc_score(y_test, proba),
            "test_f1": f1_score(y_test, pred),
            "test_acc": accuracy_score(y_test, pred),
        }
    )

final_df = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
display(final_df)

best_name = final_df.iloc[0]["model"]
best_model = final_models[best_name]
print("selected_model:", best_name)
print("confusion_matrix:\n", confusion_matrix(y_test, best_model.predict(X_test)))


,model,test_auc,test_f1,test_acc
0,log_reg,0.997694,0.988889,0.986014
3,xgboost,0.996646,0.978022,0.972028
1,rf_tuned,0.993501,0.967033,0.958042
2,grad_boost,0.992453,0.967391,0.958042


selected_model: log_reg
confusion_matrix:
 [[52  1]
 [ 1 89]]


In [6]:
def assert_selection_quality(df: pd.DataFrame, min_auc: float = 0.95) -> None:
    if df["test_auc"].max() < min_auc:
        raise ValueError(f"No model met min AUC {min_auc}")


assert_selection_quality(final_df, min_auc=0.93)
print("quality gate passed")

quality gate passed


## Business Case Studies & Exceptions
- Best-AUC model may lose if latency or governance constraints are violated.
- Leakage-safe pipelines are mandatory for trustworthy offline metrics.
- Exception: on tiny datasets, use repeated CV and report uncertainty to avoid overconfidence.

## Interview Questions & Answers
1. Why cross-validation? It reduces dependence on one split and gives more stable generalization estimates.
2. Why is data leakage dangerous? It causes inflated metrics that do not transfer to production.